# Code-Review Evaluation: does inference optimization preserve downstream quality?

Continues `llm-inference-optimizer`'s V1 -> V4 story, but swaps the generic
perplexity quality proxy for a real downstream task: **can a quantized
Qwen3-0.6B still catch bugs in code review?**, using
[`agent-review-loop`](https://github.com/sourds42/agent-review-loop)'s
golden-set tasks as the source of "diffs with known bugs," extended here
with new security- and performance-flavored examples (the source repo has
neither) and structured category/severity ground truth (which didn't
exist there either).

```
V1  FP16 baseline reviews the dataset
V2  bnb INT8 / INT4 review the SAME dataset -- one comparison table:
    config -> F1/critical-recall -> latency -> VRAM -> tokens -> cost
V3  deterministic optimizer: constraint filter + Pareto frontier +
    manual-selection heuristic, reusing src/optimizer/ unmodified
V4  agent-assisted optimizer: diagnose -> propose -> experiment loop,
    reusing src/agent/ unmodified
```

**Runtime -> Change runtime type -> T4 GPU** before running anything below.

In [ ]:
!nvidia-smi

In [ ]:
import torch

_baseline_torch_spec = torch.__version__
with open("/content/_baseline_torch_spec.txt", "w") as f:
    f.write(_baseline_torch_spec)
print("Baseline (Colab-provided) torch:", _baseline_torch_spec,
      "| CUDA available:", torch.cuda.is_available())

## Setup -- clone both repos as siblings, install, verify CUDA

In [ ]:
!git clone https://github.com/sourds42/llm-inference-optimizer.git
%cd llm-inference-optimizer

In [ ]:
# Cloned as a sibling (../agent-review-loop) -- src/review_eval/dataset.py
# looks for it there by default. agent-review-loop is never modified,
# only imported from.
!git clone https://github.com/sourds42/agent-review-loop.git ../agent-review-loop

In [ ]:
# Colab's base image sometimes preinstalls a `langchain` version that
# doesn't match the langchain-core/langgraph versions we pin, which
# crashes V4's LangGraph agent loop with
# "AttributeError: module 'langchain' has no attribute 'debug'".
!pip uninstall -y -q langchain langchain-core langchain-community langgraph 2>/dev/null

!pip install -q -r requirements.txt -r requirements-colab.txt
print("Installed.")

In [ ]:
import torch

if not torch.cuda.is_available():
    with open("/content/_baseline_torch_spec.txt") as f:
        baseline_spec = f.read().strip()
    base_version, _, cuda_tag = baseline_spec.partition("+")
    cuda_tag = cuda_tag or "cu121"
    print(f"torch.cuda.is_available() is False -- reinstalling the original Colab build ({baseline_spec})...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "--force-reinstall",
                     f"torch=={base_version}", "--index-url",
                     f"https://download.pytorch.org/whl/{cuda_tag}"], check=True)
    print("\nReinstalled. Now: Runtime -> Restart session, then re-run every cell from the nvidia-smi cell down.")
else:
    print(f"CUDA OK: {torch.cuda.get_device_name(0)}")

## Step 1 -- Build the code-review evaluation dataset

Reuses `agent-review-loop`'s 10 existing buggy/clean task pairs (all logic
bugs -- off-by-one, mutable-default, bare-except, etc.), adds 8 new
security- and performance-flavored pairs the source repo doesn't have, and
a handful of clean-only controls. Every item is fixed and versioned (see
`src/review_eval/dataset.py`), not randomly sampled, so every config below
sees exactly the same dataset.

In [ ]:
from src.review_eval.dataset import build_review_items, summarize

items = build_review_items()
summary = summarize(items)
print(f"Total review items: {summary['n_items']}  (buggy: {summary['n_buggy']}, clean: {summary['n_clean']})")
print("Buggy items by category:", summary["by_category"])
print("Buggy items by severity:", summary["by_severity"])

## Step 2 -- V1: FP16 baseline reviews the dataset

Every later config is measured against this row.

In [ ]:
from src.config import ExperimentConfig
from src.review_eval.review_task import run_review_experiment

v1_cfg = ExperimentConfig(id="review_V1__fp16_baseline", tier="review_V1",
                           quant_method="fp16", results_dir="review_results")
v1_row = run_review_experiment(v1_cfg, items)
print({k: v1_row[k] for k in ("f1", "precision", "recall", "critical_recall",
                               "severity_accuracy", "completeness", "tokens_per_sec",
                               "e2e_p95_ms", "vram_gb", "cost_usd_per_request", "error")})

## Step 3 -- V2: the already-measured quant configs, through the SAME review task

This is the actual question this notebook exists to answer: **does
quantization preserve review quality, or does it quietly get worse while
throughput/VRAM numbers look great?**

In [ ]:
import pandas as pd

configs_v2 = [
    ExperimentConfig(id="review_V2__fp16", tier="review_V2", quant_method="fp16", results_dir="review_results"),
    ExperimentConfig(id="review_V2__bnb_int8", tier="review_V2", quant_method="bnb_int8", results_dir="review_results"),
    ExperimentConfig(id="review_V2__bnb_int4", tier="review_V2", quant_method="bnb_int4", results_dir="review_results"),
]
v2_rows = [run_review_experiment(cfg, items) for cfg in configs_v2]

comparison_table = pd.DataFrame([{
    "config": r["quant_method"], "f1": r.get("f1"), "precision": r.get("precision"),
    "recall": r.get("recall"), "critical_recall": r.get("critical_recall"),
    "severity_acc": r.get("severity_accuracy"), "completeness": r.get("completeness"),
    "tokens_per_sec": r.get("tokens_per_sec"), "e2e_p95_ms": r.get("e2e_p95_ms"),
    "vram_gb": r.get("vram_gb"), "cost_usd_per_request": r.get("cost_usd_per_request"),
    "error": r.get("error"),
} for r in v2_rows])
comparison_table

**Fill in after running the cell above:** does F1/critical-recall hold up
across `bnb_int8`/`bnb_int4`, or does it drop off a cliff at some point?
Compare against V1's inference-only perplexity-recovery numbers (the
original `notebooks/run_on_colab.ipynb` run) -- does a model that "still
has 95%+ perplexity recovery" actually still review code well, or is that
proxy metric misleading for this downstream task?

## Step 4 -- V3: deterministic constraint + Pareto optimizer

Reuses `src/optimizer/` completely unmodified (aside from the one new
optional `critical_recall_min` field added to `Constraints`).
`quality_min_recovery_pct` in `configs/review_constraints.yaml` means
**F1 score \* 100** for this notebook -- the row field is populated that
way in `run_review_experiment`, reusing the existing constraint-gate code
by naming convention rather than adding a parallel schema.

In [ ]:
from src.optimizer.constraints import Constraints, filter_passing
from src.optimizer.pareto import pareto_frontier
from src.optimizer.search import best_from_frontier
from src.optimizer.manual_baseline import manual_pick

constraints = Constraints.load("configs/review_constraints.yaml")
passing = filter_passing(v2_rows, constraints)
frontier = pareto_frontier(passing)
v3_auto_best = best_from_frontier(frontier)
v3_manual_best = manual_pick(v2_rows, constraints)

print(f"{len(passing)}/{len(v2_rows)} configs pass constraints "
      f"(F1>={constraints.quality_min_recovery_pct}%, "
      f"critical_recall>={constraints.critical_recall_min}%, "
      f"P95<={constraints.p95_latency_ms_max}ms), {len(frontier)} on the Pareto frontier")
print("Automated pick:", v3_auto_best and v3_auto_best["id"])
print("Manual pick:   ", v3_manual_best and v3_manual_best["id"])
for r in v2_rows:
    reason = "PASS" if r in passing else "REJECTED"
    print(f"  {r['quant_method']:10s} F1={r.get('f1')}  crit_recall={r.get('critical_recall')}  "
          f"p95={r.get('e2e_p95_ms')}ms  -> {reason}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5))
for r in v2_rows:
    color = "#3B9EFF" if r in passing else "#888"
    ax.scatter(r.get("e2e_p95_ms", 0), (r.get("f1") or 0) * 100,
               s=(r.get("vram_gb") or 1) * 60 + 60, color=color, alpha=.7, edgecolor="k")
    ax.annotate(r["quant_method"], (r.get("e2e_p95_ms", 0), (r.get("f1") or 0) * 100), fontsize=9)
ax.set_xlabel("E2E p95 latency (ms) on the review task")
ax.set_ylabel("F1 (%) -- bug-detection quality")
ax.set_title("Code-review quality vs. latency (blue = passes constraints, size = VRAM)")
ax.grid(alpha=.3)
plt.savefig("review_results/quality_vs_latency.png", dpi=120, bbox_inches="tight")
plt.show()

**Fill in:** why was each config accepted/rejected? If nothing passes, is
it the F1 bar, the critical-recall bar, or latency that's binding --
and is `configs/review_constraints.yaml` set to a realistic target for a
0.6B reviewer, or does it need loosening for a small-model context?

## Step 5 -- V4: agent-assisted optimizer

Reuses `src/agent/build_graph`/`new_state` unmodified. Seeded with the 3
V2 results already collected (no wasted re-runs); given 2 new, genuinely
untried configs to explore so the diagnose -> propose loop has real
choices to make, not an already-exhausted search space.

In [ ]:
from src.model_client import MockClient
from src.agent.graph import build_graph, new_state

configs_v4_new = [
    ExperimentConfig(id="review_V4__bnb_int8_nocache", tier="review_V4", quant_method="bnb_int8",
                      use_cache=False, results_dir="review_results"),
    ExperimentConfig(id="review_V4__bnb_int4_ctx1024", tier="review_V4", quant_method="bnb_int4",
                      context_len=1024, results_dir="review_results"),
]
configs_by_id = {c.id: c for c in configs_v2 + configs_v4_new}

def run_experiment_for_agent(cfg):
    return run_review_experiment(cfg, items)

# MockClient by default -- free, deterministic. Set MODEL_BACKEND-style
# swap here for a real reasoning run, e.g.:
#   from src.model_client import AnthropicClient
#   model = AnthropicClient()
model = MockClient(seed=42, correct_rate=0.7)

app = build_graph(model, run_experiment_for_agent, configs_by_id, constraints)
state = new_state(initial_rows=v2_rows, all_config_ids=list(configs_by_id.keys()))
out = app.invoke(state)

print("Verdict:", out["verdict"], "| rounds:", out["round"])
for entry in out["log"]:
    print(entry)

In [ ]:
v4_passing = filter_passing(out["rows"], constraints)
v4_frontier = pareto_frontier(v4_passing)
v4_best = best_from_frontier(v4_frontier)

print("V4 agent's best config:", v4_best and v4_best["id"])
print("V3 automated best:     ", v3_auto_best and v3_auto_best["id"])
print("V3 manual best:        ", v3_manual_best and v3_manual_best["id"])

new_rows = [r for r in out["rows"] if r["id"] in out["tried_ids"]]
n_new_experiments = sum(1 for e in out["log"] if e["node"] == "experiment" and not e.get("skipped"))
print(f"New experiments run by the agent: {n_new_experiments} (out of {len(configs_v4_new)} available)")

**Fill in:** did the agent's diagnosis (`out["log"]` entries with
`node: "diagnose"`) make sense given the actual numbers? Did it find a
better config than V3's automated pick, or converge on the same one with
extra (wasted) experiments? Remember `MockClient` doesn't actually reason
-- it's a rule-of-thumb heuristic with a controllable error rate. Re-run
this cell with `AnthropicClient()` for a real-reasoning comparison if you
want to see whether reasoning actually helps here.

## Final analysis

*Fill in each subsection with the real numbers from the run above --
these are prompts, not a finished writeup.*

### Architecture
V1 (FP16 baseline reviews the dataset) -> V2 (bnb INT8/INT4 review the same
dataset, one comparison table) -> V3 (deterministic constraint + Pareto
selection, `src/optimizer/`) -> V4 (agent-assisted diagnose/propose loop,
`src/agent/`). Every stage after V1 reuses the exact same
`run_review_experiment` measurement function -- nothing about *how* a
config is scored changes between manual/automated/agent-assisted
selection, only *which* config gets tried and *who* picks the winner.

### Experiment history
How many total review-experiments were run across V1-V4? List config IDs
and their F1/critical-recall/latency/cost, pulled from
`review_results/review_experiments.jsonl`.

### Failure analysis / honest limitations
- A 0.6B model is a genuinely weak code reviewer by nature -- low absolute
  F1 here is expected and informative, not a bug.
- The dataset is single-issue-per-item (at most one bug per candidate), so
  "completeness" is a well-formedness proxy, not multi-issue coverage --
  see the docstring in `src/review_eval/metrics.py`.
- `quality_recovery_pct` = F1*100 is a naming-convention reuse of the
  original perplexity-based field, not a new schema -- worth stating
  explicitly so a reader doesn't confuse the two notebooks' numbers.
- Small, fixed dataset (~40 items) -- representative, not a statistically
  powered benchmark. Note any config that hit `error` (check
  `TROUBLESHOOTING.md` in the repo root for prior GPU/dependency issues
  and how they were fixed).
- TTFT/TPOT are approximated (separate 1-token pass, not true streaming),
  same tradeoff as `src/benchmark.py`.

### Before/after metrics
Fill in a table: V1 baseline vs. V3's automated pick vs. V4's agent pick,
across F1, critical-recall, VRAM, P50/P95 latency, tokens/sec, cost.

### Conclusions
Does inference optimization preserve downstream code-review quality on
this model/task? What would change the answer (bigger model, different
quantization method, a less single-issue-per-item dataset)? This is a
**prototype evaluation methodology**, not a production code-review
benchmark -- the architecture and discipline (deterministic gate before
LLM judgment, reused across two different downstream tasks now) is what
transfers, not these specific F1 numbers.